In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dataguruji/candidate-retrieval-data/candidate_documents.parquet
/kaggle/input/datasets/dataguruji/candidate-retrieval-data/behavior_features.parquet
/kaggle/input/datasets/dataguruji/candidate-retrieval-data/jd_hybrid_profile.pkl
/kaggle/input/datasets/dataguruji/candidate-retrieval-data/integrity_features.parquet
/kaggle/input/datasets/dataguruji/candidate-retrieval-data/candidate_metadata.parquet
/kaggle/input/datasets/dataguruji/candidate-retrieval-data/jd_profile.pkl
/kaggle/input/datasets/dataguruji/candidate-retrieval-data/jd_semantic_profile.pkl


In [2]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.3 MB/s eta 0:00:00:00:0100:01


In [21]:
import os
import gc
import pickle
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer

from rank_bm25 import BM25Okapi

In [22]:
# ============================================
# CONFIG
# ============================================

WORKING_DIR = "/kaggle/input/datasets/dataguruji/candidate-retrieval-data/"

DOCUMENT_PATH = os.path.join(
    WORKING_DIR,
    "candidate_documents.parquet"
)

JD_PATH = "/kaggle/input/datasets/dataguruji/candidate-retrieval-data/jd_hybrid_profile.pkl"

MODEL_NAME = "BAAI/bge-small-en-v1.5"

BATCH_SIZE = 256

In [23]:
candidate_docs = pd.read_parquet(DOCUMENT_PATH)

print(candidate_docs.shape)

candidate_docs.head()

(100000, 6)


,candidate_id,profile_document,career_document,skills_document,education_document,certification_document
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud Software ...",\nThe candidate currently works\nas a\n\nBacke...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.E.\n\nin\n\nComputer Science\...,
1,CAND_0000002,Operations Manager | 12.5+ yrs experience Prof...,\nThe candidate currently works\nas a\n\nOpera...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.Sc\n\nin\n\nMathematics\n\nfr...,
2,CAND_0000003,Customer Support | 1.1+ yrs experience Profess...,\nThe candidate currently works\nas a\n\nCusto...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nM.E.\n\nin\n\nChemical Engineer...,
3,CAND_0000004,Marketing Manager | Driving business outcomes ...,\nThe candidate currently works\nas a\n\nMarke...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.Tech\n\nin\n\nMachine Learnin...,\nCertified in\n\nAWS Certified Cloud Practiti...
4,CAND_0000005,Accountant | Helping teams scale Professional ...,\nThe candidate currently works\nas a\n\nAccou...,\nThe candidate has\n\nbeginner\n\nproficiency...,\nCompleted\n\nM.Sc\n\nin\n\nInformation Techn...,


In [24]:
from rank_bm25 import BM25Okapi

with open(JD_PATH, "rb") as f:
    jd_hybrid = pickle.load(f)

print(jd_hybrid.keys())

dict_keys(['raw_text', 'sentences', 'keywords', 'important_sentences', 'noun_chunks', 'entities', 'clusters', 'cluster_summary', 'cluster_embeddings', 'bm25'])


In [25]:
candidate_docs["retrieval_document"] = (

    candidate_docs["profile_document"].fillna("")

    + "\n\n"

    + candidate_docs["career_document"].fillna("")

    + "\n\n"

    + candidate_docs["skills_document"].fillna("")

    + "\n\n"

    + candidate_docs["education_document"].fillna("")

    + "\n\n"

    + candidate_docs["certification_document"].fillna("")
)

In [26]:
candidate_docs["num_words"] = (

    candidate_docs["retrieval_document"]

    .str.split()

    .str.len()

)

candidate_docs["num_words"].describe()

count    100000.000000
mean        503.438930
std         118.780182
min         247.000000
25%         416.000000
50%         499.500000
75%         582.000000
max        1111.000000
Name: num_words, dtype: float64

In [27]:
tfidf = TfidfVectorizer(

    lowercase=True,

    stop_words="english",

    max_features=120000,

    ngram_range=(1,2),

    min_df=2,

    max_df=0.90,

    sublinear_tf=True

)

tfidf_matrix = tfidf.fit_transform(

    candidate_docs["retrieval_document"]

)

print(tfidf_matrix.shape)

(100000, 12263)


In [28]:
joblib.dump(

    tfidf,

    "/kaggle/working/tfidf_vectorizer.joblib"

)

joblib.dump(

    tfidf_matrix,

    "/kaggle/working/tfidf_matrix.joblib"

)

print("TF-IDF Saved")

TF-IDF Saved


In [29]:
tokenized = [

    doc.lower().split()

    for doc in candidate_docs["retrieval_document"]

]

bm25 = BM25Okapi(tokenized)

with open(

    "/kaggle/working/bm25.pkl",

    "wb"

) as f:

    pickle.dump(

        bm25,

        f

    )

print("BM25 Saved")

BM25 Saved


In [30]:
np.save(

    "/kaggle/working/candidate_ids.npy",

    candidate_docs["candidate_id"].values

)

In [31]:
!pip install -q sentence-transformers

In [32]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded


In [34]:
import pickle

JD_PATH = "/kaggle/input/datasets/dataguruji/candidate-retrieval-data/jd_hybrid_profile.pkl"

with open(JD_PATH, "rb") as f:
    jd = pickle.load(f)

jd_query = " ".join(jd["important_sentences"])

print(jd_query[:1000])

Importance Sentence


In [35]:
jd_vector = tfidf.transform([jd_query])

tfidf_scores = (tfidf_matrix @ jd_vector.T).toarray().ravel()

tfidf_top = np.argsort(tfidf_scores)[::-1][:5000]

print(tfidf_top[:10])

[98103 19574 17454 59462 62233 70663 80348   616 96715 57202]


In [36]:
query_tokens = jd_query.lower().split()

bm25_scores = bm25.get_scores(query_tokens)

bm25_top = np.argsort(bm25_scores)[::-1][:5000]

print(bm25_top[:10])

[98103 17454 19574 96715 62233 64858 19935 70663 44637 75276]


In [37]:
from collections import defaultdict

rrf_scores = defaultdict(float)

K = 60

for rank, idx in enumerate(tfidf_top):
    rrf_scores[idx] += 1.0 / (K + rank)

for rank, idx in enumerate(bm25_top):
    rrf_scores[idx] += 1.0 / (K + rank)

rrf_sorted = sorted(
    rrf_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

In [40]:
TOP_K = 3000

candidate_indices = [

    idx

    for idx, score in rrf_sorted[:TOP_K]

]

top_candidates = candidate_docs.iloc[
    candidate_indices
].copy()

top_candidates["rrf_score"] = [

    score

    for idx, score in rrf_sorted[:TOP_K]

]

print(top_candidates.shape)

(3000, 9)


In [41]:
candidate_texts = top_candidates[
    "retrieval_document"
].tolist()

candidate_embeddings = model.encode(

    candidate_texts,

    batch_size=64,

    normalize_embeddings=True,

    convert_to_numpy=True,

    show_progress_bar=True

)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

In [42]:
jd_embedding = model.encode(

    jd_query,

    normalize_embeddings=True,

    convert_to_numpy=True
)

In [43]:
semantic_scores = candidate_embeddings @ jd_embedding

top_candidates["semantic_score"] = semantic_scores

In [44]:
top_candidates["retrieval_score"] = (

    0.60 * top_candidates["semantic_score"]

    + 0.25 * top_candidates["rrf_score"]

    + 0.15 * (
        top_candidates["rrf_score"]
        / top_candidates["rrf_score"].max()
    )

)

In [45]:
top_candidates.to_parquet(

    "/kaggle/working/top3000_candidates.parquet",

    index=False

)

print("Saved Successfully")

Saved Successfully


In [46]:
import faiss
import numpy as np
import os
import pickle

In [47]:
embeddings = np.load(
    "/kaggle/working/candidate_embeddings.npy"
).astype(np.float32)

candidate_ids = np.load(
    "/kaggle/working/candidate_ids.npy",
    allow_pickle=True
)

print(embeddings.shape)
print(candidate_ids.shape)

(100000, 384)
(100000,)


In [48]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors in FAISS:", index.ntotal)

Vectors in FAISS: 100000


In [51]:
faiss.write_index(

    index,

    "/kaggle/working/candidate.index"

)

print("FAISS Saved Successfully")

FAISS Saved Successfully


In [52]:
index = faiss.read_index(
    "/kaggle/working/candidate.index"
)

print(index.ntotal)

100000


In [53]:
metadata = {

    "embedding_model":"BAAI/bge-small-en-v1.5",

    "embedding_dimension":embeddings.shape[1],

    "num_candidates":len(candidate_ids),

    "faiss_metric":"Inner Product",

    "normalize_embeddings":True

}

with open(

    "/kaggle/working/retrieval_metadata.pkl",

    "wb"

) as f:

    pickle.dump(

        metadata,

        f

    )

print("Metadata Saved")

Metadata Saved


In [54]:
required = [

    "candidate_embeddings.npy",

    "candidate_ids.npy",

    "candidate.index",

    "tfidf_vectorizer.joblib",

    "tfidf_matrix.joblib",

    "bm25.pkl",

    "retrieval_metadata.pkl"

]

print("="*60)

for file in required:

    path = os.path.join("/kaggle/working", file)

    if os.path.exists(path):

        print(f"✅ {file}")

    else:

        print(f"❌ {file}")

print("="*60)

✅ candidate_embeddings.npy
✅ candidate_ids.npy
✅ candidate.index
✅ tfidf_vectorizer.joblib
✅ tfidf_matrix.joblib
✅ bm25.pkl
✅ retrieval_metadata.pkl


In [55]:
import pandas as pd

df = pd.read_parquet("top3000_candidates.parquet")
print(df.columns.tolist())
print(df.head())

['candidate_id', 'profile_document', 'career_document', 'skills_document', 'education_document', 'certification_document', 'retrieval_document', 'num_words', 'rrf_score', 'semantic_score', 'retrieval_score']
   candidate_id                                   profile_document  \
0  CAND_0098104  Java Developer | Full-stack development Softwa...   
1  CAND_0019575  Frontend Engineer | Cloud & DevOps Software en...   
2  CAND_0017455  Full Stack Developer | Cloud & DevOps Software...   
3  CAND_0062234  Cloud Engineer | Backend systems & APIs Softwa...   
4  CAND_0096716  QA Engineer | Full-stack development Software ...   

                                     career_document  \
0  \nThe candidate currently works\nas a\n\nJava ...   
1  \nThe candidate currently works\nas a\n\nFront...   
2  \nThe candidate currently works\nas a\n\nFull ...   
3  \nThe candidate currently works\nas a\n\nCloud...   
4  \nThe candidate currently works\nas a\n\nQA En...   

                                  